# 05 - Decomposicao Sazonal de Series Temporais

## Objetivo
Decompor series em componentes: tendencia, sazonal e residual.

## Fluxo
1. Carregar dados
2. Decomposicao de series temporais
3. Analise de tendencia
4. Padroes sazonais
5. Residuos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from statsmodels.tsa.seasonal import seasonal_decompose

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 10)

## 1. Carregar dados

In [ ]:
caminho_dados = Path('../dados/brutos/indicadores_consolidados.csv')
df = pd.read_csv(caminho_dados)
df['data'] = pd.to_datetime(df['data'])
df = df.sort_values('data').reset_index(drop=True)
df.set_index('data', inplace=True)

print(f'Dados carregados: {len(df)} observacoes')

## 2. Decomposicao Sazonal - IPCA Mensal

In [ ]:
# Decomposicao aditiva (IPCA mensal)
decomp_ipca = seasonal_decompose(df['ipca_mensal'], model='additive', period=12)

# Plotar decomposicao
fig = plt.figure(figsize=(14, 12))

ax1 = plt.subplot(4, 1, 1)
ax1.plot(decomp_ipca.observed, linewidth=2, color='black')
ax1.set_ylabel('Original')
ax1.set_title('Decomposicao Sazonal - IPCA Mensal', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2 = plt.subplot(4, 1, 2)
ax2.plot(decomp_ipca.trend, linewidth=2, color='#1f77b4')
ax2.set_ylabel('Tendencia')
ax2.grid(True, alpha=0.3)

ax3 = plt.subplot(4, 1, 3)
ax3.plot(decomp_ipca.seasonal, linewidth=2, color='#ff7f0e')
ax3.set_ylabel('Sazonal')
ax3.grid(True, alpha=0.3)

ax4 = plt.subplot(4, 1, 4)
ax4.plot(decomp_ipca.resid, linewidth=1.5, color='#2ca02c', alpha=0.7)
ax4.axhline(y=0, color='red', linestyle='--', linewidth=1)
ax4.set_ylabel('Residuo')
ax4.set_xlabel('Data')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Analise da Tendencia

In [ ]:
trend = decomp_ipca.trend.dropna()

print('ANALISE DA TENDENCIA (TREND):')
print('=' * 50)
print(f'Valor inicial: {trend.iloc[0]:.3f}%')
print(f'Valor final: {trend.iloc[-1]:.3f}%')
print(f'Mudanca absoluta: {trend.iloc[-1] - trend.iloc[0]:+.3f}%')
print(f'Media da tendencia: {trend.mean():.3f}%')
print()

# Verificar tendencia geral
primeira_metade = trend.iloc[:len(trend)//2].mean()
segunda_metade = trend.iloc[len(trend)//2:].mean()
print(f'Media primeira metade: {primeira_metade:.3f}%')
print(f'Media segunda metade: {segunda_metade:.3f}%')
print(f'Diferenca: {segunda_metade - primeira_metade:+.3f}%')

## 4. Padroes Sazonais

In [ ]:
seasonal = decomp_ipca.seasonal.dropna().unique()[:12]
meses = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

print('COMPONENTE SAZONAL POR MES:')
print('=' * 50)
for mes, valor in zip(meses, seasonal):
    print(f'{mes}: {valor:+.3f}%')

print()

# Meses com maior sazonalidade positiva/negativa
idx_max = np.argmax(seasonal)
idx_min = np.argmin(seasonal)

print(f'Maior pressao sazonal: {meses[idx_max]} ({seasonal[idx_max]:+.3f}%)')
print(f'Menor pressao sazonal: {meses[idx_min]} ({seasonal[idx_min]:+.3f}%)')

## 5. Grafico de Sazonalidade

In [ ]:
plt.figure(figsize=(12, 6))
cores = ['#d62728' if x < 0 else '#2ca02c' for x in seasonal]
bars = plt.bar(meses, seasonal, color=cores, alpha=0.7, edgecolor='black', linewidth=1.5)
plt.axhline(y=0, color='black', linewidth=1)
plt.title('Componente Sazonal - IPCA Mensal', fontsize=14, fontweight='bold')
plt.ylabel('Impacto Sazonal (%)')
plt.grid(True, alpha=0.3, axis='y')

# Adicionar valores nas barras
for i, (bar, val) in enumerate(zip(bars, seasonal)):
    plt.text(i, val + (0.01 if val > 0 else -0.02), f'{val:.3f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Analise de Residuos

In [ ]:
residuos = decomp_ipca.resid.dropna()

print('ANALISE DE RESIDUOS:')
print('=' * 50)
print(f'Media dos residuos: {residuos.mean():.6f}%')
print(f'Desvio Padrao: {residuos.std():.3f}%')
print(f'Valor maximo: {residuos.max():+.3f}%')
print(f'Valor minimo: {residuos.min():+.3f}%')
print()

# Residuos fora de 2 desvios padroes
limite = 2 * residuos.std()
outliers_residuo = residuos[(residuos > limite) | (residuos < -limite)]
print(f'Residuos extremos (>2 sigma): {len(outliers_residuo)}')

## 7. Boxplot dos Residuos

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
sns.boxplot(y=residuos, ax=ax1, palette='Set2')
ax1.set_title('Boxplot dos Residuos', fontsize=12, fontweight='bold')
ax1.set_ylabel('Residuo (%)')
ax1.grid(True, alpha=0.3, axis='y')

# Histograma
ax2.hist(residuos, bins=15, color='#1f77b4', alpha=0.7, edgecolor='black')
ax2.axvline(x=residuos.mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {residuos.mean():.6f}')
ax2.set_title('Distribuicao dos Residuos', fontsize=12, fontweight='bold')
ax2.set_xlabel('Residuo (%)')
ax2.set_ylabel('Frequencia')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 8. Resumo da Decomposicao

In [ ]:
print('
RESUMO DA DECOMPOSICAO SAZONAL:')
print('=' * 60)
print()
print('COMPONENTES:')
print(f'  Observado: Valor original de IPCA mensal')
print(f'  Tendencia: Movimento de longo prazo (variacao: {trend.min():.3f}% a {trend.max():.3f}%)')
print(f'  Sazonal: Padrao repetitivo mensal (variacao: {seasonal.min():+.3f}% a {seasonal.max():+.3f}%)')
print(f'  Residual: Componente irregular (std: {residuos.std():.3f}%)')
print()
print('INTERPRETACAO:')
print(f'  - Tendencia geral: {"Crescente" if trend.iloc[-1] > trend.iloc[0] else "Decrescente"}')
print(f'  - Sazonalidade forte em: {meses[np.argmax(np.abs(seasonal))]}')
print(f'  - Residuos: {"Bem comportados" if residuos.std() < 0.1 else "Com variabilidade significativa"}')